# Tydzień 4: Regresja – strojenie modelu, predykcje i zapis do pliku 🎛️💾

| | |
|---|---|
| **Rozdział książki** | 2. *Regression* – podrozdziały: *Creating the Model*, *Tuning the Model*, *Making Predictions*, *Plotting the Model*, *Finalizing and Saving the Model* |
| **Czas zajęć** | 90 minut |
| **Dane** | `insurance` – koszty ubezpieczenia zdrowotnego (wbudowane w PyCaret) |
| **Nowe funkcje PyCaret** | `create_model()`, `tune_model()`, `predict_model()`, `plot_model()`, `finalize_model()`, `save_model()`, `load_model()` |

## 🎯 Cel zajęć
Po dzisiejszych zajęciach będziesz:
1. umieć zbudować wybrany model funkcją `create_model()` i odczytać wyniki walidacji krzyżowej,
2. rozumieć różnicę między **parametrami** a **hiperparametrami** i umieć **dostroić** model (`tune_model`),
3. sprawdzić model na **zbiorze testowym** i narysować jego błędy oraz **ważność cech**,
4. **zapisać** gotowy model do pliku, **wczytać** go z powrotem i przewidzieć koszt dla nowej osoby.

## 🗓️ Plan zajęć
| Czas | Co robimy |
|---|---|
| 0–15 min | 📚 Teoria: hiperparametry, strojenie, finalizacja |
| 15–65 min | 👣 Krok po kroku: od `create_model()` do `save_model()` |
| 65–85 min | 🧑‍💻 Ćwiczenie: ile kosztuje palenie? – ✅ **obowiązkowe** (zaliczenie) |
| 85–90 min | 📝 Podsumowanie |
| poza zajęciami | 🏋️ Część 4: ćwiczenia dodatkowe (trudniejsze, nieobowiązkowe) – dla szybszych osób lub do domu |

---
# 📚 Część 1: Wprowadzenie teoretyczne

## Parametry vs. hiperparametry 🎛️
- **Parametry** (*parameters*) – model uczy się ich **sam** z danych (np. współczynniki β w regresji liniowej).
- **Hiperparametry** (*hyperparameters*) – ustawiamy je **my, przed treningiem**. To jak pokrętła w piekarniku:
  temperatura i czas pieczenia. Ciasto „samo” się upiecze, ale od ustawień zależy, czy wyjdzie dobre.

Przykładowe hiperparametry modelu *Gradient Boosting*:
| Hiperparametr | Co oznacza (po ludzku) |
|---|---|
| `n_estimators` | ile małych drzew decyzyjnych budujemy (każde poprawia błędy poprzednich) |
| `max_depth` | jak „głębokie” (szczegółowe) może być każde drzewo |
| `learning_rate` | jak duże „kroki” robi model przy poprawianiu błędów |
| `subsample` | jaką część danych widzi każde drzewo (np. 0.8 = 80%) |

## Strojenie hiperparametrów (*Hyperparameter Tuning*) 🔍
Zamiast zgadywać ustawienia, sprawdzamy wiele kombinacji i wybieramy najlepszą.
`tune_model()` domyślnie używa **przeszukiwania losowego** (*random search*): losuje `n_iter` kombinacji z podanej siatki
(*grid*) i każdą ocenia walidacją krzyżową. To jak próbowanie losowych ustawień piekarnika zamiast wszystkich możliwych.

⚠️ Strojenie bywa **wolne** (model trenuje się wiele razy). Gdy trwa za długo – zmniejsz `n_iter`.

## Przeuczenie (*Overfitting*) 📚
Uczeń, który **wkuł na pamięć** odpowiedzi z ćwiczeń, oblewa egzamin z nowymi zadaniami. Model też może się „przeuczyć”:
świetnie działa na danych treningowych, a słabo na nowych. Dlatego **ostateczny sprawdzian** robimy na zbiorze testowym (`predict_model`).

## Finalizacja i zapis modelu 💾
- `finalize_model()` – gdy jesteśmy zadowoleni z modelu, trenujemy go jeszcze raz na **wszystkich** danych (trening + test),
  żeby wykorzystał każdą informację.
- `save_model()` – „zamraża” model (razem z całym potokiem przygotowania danych!) do pliku `.pkl`.
  Jak obiad zamrożony w słoiku 🫙 – można go później rozmrozić (`load_model()`) i od razu użyć, bez ponownego gotowania.

---
# 👣 Część 2: Krok po kroku

### Krok 1: Powtórka – dane i `setup()` (tak jak tydzień temu)
Każdy notebook działa samodzielnie, więc musimy ponownie przygotować środowisko PyCaret.

In [ ]:
# Wyłączamy ostrzeżenia (warnings), żeby nie zaśmiecały ekranu.
# Ostrzeżenie to NIE błąd – to tylko podpowiedź dla programistów bibliotek.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Importy: pandas (tabele), get_data (dane), wszystkie funkcje regresji PyCaret
import pandas as pd
from pycaret.datasets import get_data
from pycaret.regression import *

In [ ]:
# Wczytujemy dane; verbose=False wyłącza wyświetlanie tabeli
data = get_data('insurance', verbose=False)

In [ ]:
# Ten sam setup co w tygodniu 3 (verbose=False ukrywa tabelę ustawień – znamy ją już)
reg = setup(data=data, target='charges', train_size=0.8, session_id=7402,
            numeric_features=['age', 'bmi', 'children'], categorical_features=['smoker', 'sex', 'region'],
            transformation=True, normalize=True, verbose=False)

### Krok 2: Budujemy model – `create_model()` 🏗️
W zeszłym tygodniu `compare_models()` wskazał zwycięzcę: **Gradient Boosting Regressor** (ID: `'gbr'`).
Teraz zbudujemy tylko ten jeden model. Tabela pokaże wynik dla **każdej z 10 części** walidacji krzyżowej.

In [ ]:
# 'gbr' = Gradient Boosting Regressor; fold=10 – walidacja krzyżowa na 10 częściach
model = create_model('gbr', fold=10)

📖 **Jak czytać tabelę?** Wiersze `0–9` to wyniki na kolejnych częściach (*folds*).
- `Mean` – **średnia** z 10 prób → to nasz główny wynik,
- `Std` – **odchylenie standardowe**: jak bardzo wyniki „skaczą” między częściami (mniejsze = stabilniejszy model).

In [ ]:
# Aktualne hiperparametry modelu – "pozycje pokręteł" przed strojeniem
model.get_params()

### Krok 3: Strojenie hiperparametrów – `tune_model()` 🎛️
Przygotowujemy **siatkę** (*grid*) – słownik: nazwa hiperparametru → lista wartości do sprawdzenia.

In [ ]:
params = {
    'learning_rate': [0.05, 0.1],        # wielkość "kroku" uczenia
    'max_depth': [2, 3, 4, 5],           # maksymalna głębokość drzew
    'n_estimators': [100, 200, 300],     # liczba drzew
    'subsample': [0.8, 1.0],             # część danych widziana przez każde drzewo
}

In [ ]:
# Losujemy 15 kombinacji (n_iter=15) i każdą sprawdzamy 5-krotną walidacją krzyżową (fold=5).
# optimize='RMSE' – szukamy ustawień z najmniejszym RMSE. ⏳ ok. 1 minuty
tuned_model = tune_model(model, optimize='RMSE', fold=5, custom_grid=params, n_iter=15)

In [ ]:
# Najlepsze znalezione ustawienia
print(tuned_model)

💡 Porównaj wiersz `Mean` przed strojeniem (krok 2) i po strojeniu (powyżej). Zwykle RMSE trochę spada.
Uwaga: PyCaret jest „ostrożny” – jeśli dostrojony model okaże się **gorszy** od oryginału, `tune_model()` zwróci oryginał
(parametr `choose_better=True`). Nie da się więc „zepsuć” modelu strojeniem.

### Krok 4: Egzamin na zbiorze testowym – `predict_model()` 📝
Do tej pory oceniał nas tylko zbiór treningowy (walidacja krzyżowa). Teraz sprawdzamy model na **20% danych testowych**,
których model **nigdy nie widział**.

In [ ]:
# Bez parametru data= funkcja używa zbioru testowego przygotowanego w setup()
predictions = predict_model(tuned_model)

In [ ]:
# Tabela z predykcjami: oryginalne kolumny + 'charges' (prawdziwy koszt) + 'prediction_label' (przewidywanie modelu)
predictions.head()

In [ ]:
# Dodajemy kolumnę z błędem: o ile dolarów model się pomylił (przewidywanie − prawda)
predictions['error'] = predictions['prediction_label'] - predictions['charges']
predictions[['charges', 'prediction_label', 'error']].head(10)

🔎 Jeśli RMSE na zbiorze testowym jest **podobne** (albo nawet lepsze) niż w walidacji krzyżowej – świetnie,
model **nie jest przeuczony** i dobrze radzi sobie z nowymi danymi.

### Krok 5: Wykresy modelu – `plot_model()` 📈

In [ ]:
# Ważność cech (feature importance): które cechy najbardziej pomagają w przewidywaniu?
plot_model(tuned_model, plot='feature')

🔎 Zdecydowanie najważniejsze jest **palenie** (`smoker`), a za nim **BMI** (`bmi`) i **wiek** (`age`) – zgodnie z EDA z tygodnia 2! 🎉
BMI jest tak wysoko, bo model „odkrył” interakcję, którą widzieliśmy na wykresach: palacze z wysokim BMI płacą najwięcej.
Płeć i region prawie nie mają znaczenia.

In [ ]:
# Wykres błędu predykcji: oś X = prawdziwa wartość (y), oś Y = przewidywanie (ŷ).
# Idealny model miałby wszystkie kropki na szarej przerywanej linii "identity".
plot_model(tuned_model, plot='error')

In [ ]:
# Wykres reszt (residuals): reszta = prawda − przewidywanie. Dobrze, gdy kropki są rozrzucone wokół zera bez wyraźnego wzoru.
plot_model(tuned_model, plot='residuals')

🔎 Większość kropek leży blisko linii idealnej. Kilka kropek **poniżej** linii to osoby, którym model **zaniżył** koszt –
to prawdopodobnie przypadki, których nie da się przewidzieć z tych kilku cech (np. nagła choroba).

### Krok 6: Finalizacja i zapis modelu – `finalize_model()` + `save_model()` 💾

In [ ]:
# Trenujemy dostrojony model na WSZYSTKICH danych (trening + test)
final_model = finalize_model(tuned_model)

In [ ]:
# Tworzymy folder "modele" obok folderu "notebooks" (exist_ok=True – nie zgłaszaj błędu, jeśli już istnieje)
import os
os.makedirs('../modele', exist_ok=True)

In [ ]:
# Zapisujemy model (z całym potokiem przygotowania danych) do pliku ../modele/regression_model.pkl
# Średnik ; na końcu ukrywa bardzo długi opis potoku, który funkcja zwraca.
save_model(final_model, '../modele/regression_model');

📁 W panelu plików po lewej stronie powinien pojawić się plik `modele/regression_model.pkl`.
Wykorzystamy go w tygodniu 12 w aplikacji internetowej!

### Krok 7: Wczytanie modelu i przewidywanie dla nowej osoby 🔮
Wyobraź sobie, że jesteś agentem ubezpieczeniowym i przychodzi nowy klient.

In [ ]:
# Wczytujemy ("rozmrażamy") zapisany model – .pkl dopisze się automatycznie
loaded_model = load_model('../modele/regression_model')

In [ ]:
# Nowa osoba – tabela z JEDNYM wierszem. Kolumny takie jak w danych, ale BEZ 'charges' (to chcemy przewidzieć!)
new_person = pd.DataFrame([{'age': 35, 'sex': 'male', 'bmi': 27.5, 'children': 1,
                            'smoker': 'no', 'region': 'northwest'}])
new_person

In [ ]:
# Przewidywanie: wynik jest w kolumnie 'prediction_label'
predict_model(loaded_model, data=new_person)

---
# 🧑‍💻 Część 3: Ćwiczenie dla Ciebie
> ✅ **Ćwiczenie obowiązkowe** – na jego podstawie zaliczasz dzisiejsze zajęcia (zaliczone / niezaliczone).

**Zadanie: Ile kosztuje palenie?** 🚬💸

- **a)** Utwórz tabelę `two_people` z **dwiema** osobami, które różnią się **tylko** paleniem:
  kobieta, 45 lat, BMI 31, 2 dzieci, region `southeast` – raz `smoker='no'`, raz `smoker='yes'`.
- **b)** Przewidź koszty dla obu osób wczytanym modelem (`loaded_model`).
- **c)** Oblicz różnicę między przewidywanymi kosztami. Ile razy drożej płaci osoba paląca?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Tabela z dwoma wierszami: `pd.DataFrame([{...osoba 1...}, {...osoba 2...}])` – lista dwóch słowników.
- Wynik `predict_model(...)` zapisz do zmiennej, np. `result`. Kolumna z przewidywaniem to `result['prediction_label']`.
- Pierwszy wiersz to `.iloc[0]`, drugi – `.iloc[1]`.

</details>

In [ ]:
# ✍️ Twój kod tutaj (możesz dodawać kolejne komórki przyciskiem "+ Code")

### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

🌟 **Dla chętnych:** sprawdź, jak zmienia się przewidywany koszt dla tej samej osoby w wieku 25, 45 i 65 lat
(trzy wiersze w tabeli, zmienia się tylko `age`).

---
# 🏋️ Część 4: Ćwiczenia dodatkowe – trochę trudniejsze
Dla osób, które skończyły ćwiczenie podstawowe, albo **do samodzielnej pracy w domu**.
Ćwiczenia z tej części są **nieobowiązkowe** – nie są wymagane do zaliczenia zajęć.

Te zadania mają mniej podpowiedzi w treści, łączą kilka poznanych narzędzi i czasem wprowadzają **jedną nową funkcję**
(zawsze opisaną w podpowiedzi). Poziomy trudności: ⭐⭐ – średnie, ⭐⭐⭐ – wyzwanie.

## ⭐⭐ Ćwiczenie 2: Pojedynek – dostrojony las losowy kontra Gradient Boosting
- **a)** Zbuduj model **Random Forest** (`'rf'`) i dostrój go własną siatką hiperparametrów:
  `n_estimators` ∈ {100, 200, 300}, `max_depth` ∈ {3, 4, 5, 6, None}, `min_samples_leaf` ∈ {1, 2, 5, 10}
  (10 losowych kombinacji, 5-krotna walidacja, optymalizacja RMSE).
- **b)** Sprawdź **oba** dostrojone modele (`tuned_model` i nowy las) na **zbiorze testowym** i zestaw ich wyniki (MAE, RMSE, R²) w jednej tabeli.
- **c)** Który model wygrał na zbiorze testowym? Czy różnica jest duża?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Hiperparametr `min_samples_leaf` to minimalna liczba osób w „liściu” drzewa – większe wartości dają prostsze drzewa.
  `None` przy `max_depth` oznacza „bez limitu głębokości”.
- `tune_model(rf, optimize='RMSE', fold=5, n_iter=10, custom_grid=...)`.
- Po `predict_model(model)` funkcja `pull()` zwraca **jednowierszową** tabelę z metrykami na zbiorze testowym.
- Dwie tabele łączy `pd.concat([tabela1, tabela2])`.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

## ⭐⭐⭐ Ćwiczenie 3: Analiza błędów – kiedy model się myli?
Dobry analityk nie poprzestaje na jednej liczbie (RMSE) – sprawdza, **gdzie** model popełnia największe błędy.
Użyj tabeli `predictions` z kroku 4 (predykcje dostrojonego Gradient Boosting na zbiorze testowym).

- **a)** Dodaj kolumnę `abs_error` z **bezwzględnym** błędem (bez znaku minus).
- **b)** Wyświetl **5 osób z największym błędem**. Co mają wspólnego? Czy model koszt **zawyżył**, czy **zaniżył**?
- **c)** Policz średni bezwzględny błąd osobno dla palących i niepalących.
- **d)** Jak myślisz – dlaczego tych kilku osób model nie mógł dobrze przewidzieć?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Wartość bezwzględna: `.abs()`, np. `predictions['error'].abs()`.
- `tabela.nlargest(5, 'abs_error')` zwraca 5 wierszy z największymi wartościami w kolumnie.
- Grupowanie: `predictions.groupby('smoker')['abs_error'].mean()`.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 📝 Podsumowanie
- `create_model('gbr')` buduje jeden model i ocenia go walidacją krzyżową.
- **Hiperparametry** ustawiamy przed treningiem; `tune_model()` szuka ich najlepszej kombinacji (*random search*).
- `predict_model()` bez danych = egzamin na **zbiorze testowym**; z `data=...` = przewidywanie dla nowych obserwacji.
- `plot_model()` rysuje m.in. ważność cech (`'feature'`), błąd predykcji (`'error'`), reszty (`'residuals'`).
- `finalize_model()` trenuje na wszystkich danych, `save_model()` / `load_model()` zapisują i wczytują model z pliku `.pkl`.

## 📖 Słowniczek PL / EN
| Polski | English | Znaczenie |
|---|---|---|
| hiperparametr | hyperparameter | ustawienie modelu wybierane przed treningiem |
| strojenie hiperparametrów | hyperparameter tuning | szukanie najlepszych ustawień modelu |
| przeszukiwanie losowe | random search | sprawdzanie losowych kombinacji ustawień |
| siatka parametrów | parameter grid | lista wartości do sprawdzenia |
| przeuczenie | overfitting | model „wkuł” dane treningowe i słabo uogólnia |
| ważność cech | feature importance | jak bardzo każda cecha pomaga modelowi |
| reszta | residual | różnica: prawdziwa wartość − przewidywanie |
| finalizacja modelu | model finalization | ponowny trening na wszystkich danych |
| serializacja / plik pickle | serialization / pickle file (.pkl) | zapis obiektu Pythona do pliku |

➡️ **Za tydzień:** zaczynamy **klasyfikację** – będziemy rozpoznawać gatunki irysów 🌸.